# Customer Behavior Analysis

This notebook focuses on analyzing customer purchasing patterns and behaviors including:
1. Customer Segmentation
2. Purchase Frequency Analysis
3. Regional Preferences
4. Customer Loyalty Patterns

## Setup and Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from datetime import datetime
from sklearn.cluster import KMeans

plt.style.use('seaborn')
%matplotlib inline

## 1. Data Loading and Customer Metrics Calculation

In [ ]:
# Load the data
df = pd.read_csv('../data/raw/sales_data.csv')
df['Date'] = pd.to_datetime(df['Date'])

# Calculate customer metrics
customer_metrics = df.groupby('Customer_ID').agg({
    'Date': ['count', 'min', 'max'],  # Purchase frequency and first/last purchase
    'Total_Amount': ['sum', 'mean'],  # Total spend and average transaction value
    'Discount': 'mean'  # Average discount received
}).round(2)

customer_metrics.columns = ['purchase_count', 'first_purchase', 'last_purchase',
                          'total_spend', 'avg_transaction', 'avg_discount']

# Calculate days between first and last purchase
customer_metrics['days_active'] = (customer_metrics['last_purchase'] - 
                                  customer_metrics['first_purchase']).dt.days

customer_metrics.head()

## 2. Customer Segmentation

In [ ]:
# Prepare data for clustering
clustering_data = customer_metrics[['total_spend', 'purchase_count', 'avg_transaction']].copy()
clustering_data = (clustering_data - clustering_data.mean()) / clustering_data.std()

# Perform K-means clustering
kmeans = KMeans(n_clusters=3, random_state=42)
customer_metrics['segment'] = kmeans.fit_predict(clustering_data)

# Visualize segments
plt.figure(figsize=(10, 6))
sns.scatterplot(data=customer_metrics, x='total_spend', y='purchase_count', 
                hue='segment', palette='deep')
plt.title('Customer Segments based on Spending and Purchase Frequency')
plt.show()

## 3. Regional Analysis

In [ ]:
# Regional preferences
regional_prefs = df.groupby(['Region', 'Category'])['Total_Amount'].sum().unstack()

# Create heatmap
plt.figure(figsize=(12, 6))
sns.heatmap(regional_prefs, annot=True, fmt='.0f', cmap='YlOrRd')
plt.title('Regional Category Preferences (Total Sales)')
plt.show()

# Regional payment methods
regional_payments = pd.crosstab(df['Region'], df['Payment_Method'], 
                               values=df['Total_Amount'], aggfunc='sum')

# Create stacked bar chart
regional_payments.plot(kind='bar', stacked=True, figsize=(10, 6))
plt.title('Payment Methods by Region')
plt.xlabel('Region')
plt.ylabel('Total Sales ($)')
plt.legend(title='Payment Method', bbox_to_anchor=(1.05, 1))
plt.tight_layout()
plt.show()

## 4. Purchase Patterns Analysis

In [ ]:
# Customer purchase patterns over time
daily_customers = df.groupby('Date')['Customer_ID'].nunique()

plt.figure(figsize=(12, 6))
daily_customers.plot()
plt.title('Daily Active Customers')
plt.xlabel('Date')
plt.ylabel('Number of Unique Customers')
plt.grid(True)
plt.show()

# Average basket size by customer segment
customer_segments = pd.merge(df, 
                            customer_metrics['segment'], 
                            left_on='Customer_ID', 
                            right_index=True)

plt.figure(figsize=(10, 6))
sns.boxplot(x='segment', y='Quantity', data=customer_segments)
plt.title('Basket Size by Customer Segment')
plt.show()

## 5. Customer Loyalty Analysis

In [ ]:
# Calculate customer lifetime value (CLV)
customer_metrics['clv'] = customer_metrics['total_spend'] / customer_metrics['days_active']
customer_metrics['clv'] = customer_metrics['clv'].fillna(customer_metrics['total_spend'])

# Plot CLV distribution
plt.figure(figsize=(10, 6))
sns.histplot(data=customer_metrics, x='clv', bins=30)
plt.title('Customer Lifetime Value Distribution')
plt.xlabel('Daily Customer Value ($)')
plt.show()

# Top 10 most valuable customers
print("Top 10 Most Valuable Customers:")
customer_metrics.nlargest(10, 'clv')[['total_spend', 'purchase_count', 'clv']]

## 6. Key Insights

1. Customer Segments:
   - Identified distinct customer segments based on spending patterns
   - Analyzed segment-specific behaviors and preferences

2. Regional Patterns:
   - Mapped regional category preferences
   - Analyzed payment method variations by region

3. Purchase Behavior:
   - Tracked daily customer activity
   - Analyzed basket sizes across segments

4. Customer Loyalty:
   - Calculated and analyzed Customer Lifetime Value
   - Identified most valuable customers

5. Recommendations:
   - Targeted marketing strategies for each segment
   - Region-specific product offerings
   - Loyalty program optimization opportunities